In [0]:
# Installing MissForest
%pip install missforest
%pip install lightgbm
%pip install tqdm

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
# Importing Librarires
import warnings
import pandas as pd
import numpy as np
from missforest import MissForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [0]:
# Loading the dataset
df = spark.table("workspace.default.credit_card_clients").drop("ID").toPandas()
print(df.head())

   LIMIT_BAL  SEX  EDUCATION  ...  PAY_AMT5  PAY_AMT6  default payment next month
0      20000    2          2  ...         0         0                           1
1     120000    2          2  ...         0      2000                           1
2      90000    2          2  ...      1000      5000                           0
3      50000    2          2  ...      1069      1000                           0
4      50000    1          2  ...       689       679                           0

[5 rows x 24 columns]


In [0]:
# Setting target variable
y = df['default payment next month']
x = df.drop('default payment next month', axis=1)

#train/test split with a ratio of 85/15
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.15, random_state= 31, stratify= y)

In [0]:
# There are 43 duplicated entries so they will be dropped
print(f'Number of duplicated entries: {x_train.duplicated().sum()}')
duplicates_mask = ~x_train.duplicated()
x_train = x_train[duplicates_mask].reset_index(drop=True)
y_train = y_train[duplicates_mask].reset_index(drop=True)

Number of duplicated entries: 43


In [0]:
# According to the documentation of the dataset
# Education (1 = graduate school; 2 = university; 3 = high school; 4 = others).
# Marriage(1 = married; 2 = single; 3 = others).
# But there are a few observations that are not documented, and therefore will be considerd as Nulls
# We will use MissForest to figure out their real value
undocumented = {
    'EDUCATION': {0: np.nan, 5: np.nan, 6: np.nan},
    'MARRIAGE': {0: np.nan}
    }
x_train.replace(undocumented, inplace= True)
x_test.replace(undocumented, inplace= True)

In [0]:
# MissForest imputation
cat_vars = ['SEX', 'EDUCATION', 'MARRIAGE']

missforest_imputer = MissForest(clf= RandomForestClassifier(n_estimators=100, max_depth= 10, random_state= 67), categorical= cat_vars)

x_train = pd.DataFrame(missforest_imputer.fit_transform(x_train), columns=x.columns)
x_test = pd.DataFrame(missforest_imputer.transform(x_test), columns=x.columns)

100%|██████████| 5/5 [00:00<00:00, 45.75it/s]


In [0]:
# Verifying the imputation worked
print(x_train['EDUCATION'].value_counts())
print(x_train['MARRIAGE'].value_counts())

# Checking no NaNs remain
nan_check = x_train[['EDUCATION', 'MARRIAGE']].isna().sum()
print(f'NANs check: {nan_check}')

2.0    12125
1.0     9034
3.0     4196
4.0      102
Name: EDUCATION, dtype: int64
2.0    13578
1.0    11589
3.0      290
Name: MARRIAGE, dtype: int64
NANs check: EDUCATION    0
MARRIAGE     0
dtype: int64


In [0]:
# 1. Re-combine X and y into one dataframe to save
train_df = x_train.copy()
train_df['default_payment_next_month'] = y_train
train_df['split'] = 'train' 

test_df = x_test.copy()
test_df['default_payment_next_month'] = y_test
test_df['split'] = 'test' 

master_df = pd.concat([train_df, test_df], axis=0)
spark.createDataFrame(master_df).write.mode("overwrite").saveAsTable("workspace.default.credit_model_data")